In [ ]:
import json
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import torch

# 路径
INPUT_DIR = Path(r".\7-1-knowledge_base_output")



device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_path = r"D:\huggingface_models\BAAI_bge_base_en_v15"

model = SentenceTransformer(
    model_path,
    device=device,
    local_files_only=True
)

Using device: cuda


In [7]:
def extract_text(record):
    # 兼容不同 JSONL 字段名
    for key in ["content", "text", "summary", "answer", "description"]:
        if record.get(key):
            return str(record[key])
    return json.dumps(record, ensure_ascii=False)

docs = []
for jsonl_file in INPUT_DIR.glob("*.jsonl"):
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except Exception:
                continue

            text = extract_text(record)
            doc_id = record.get("id") or f"{jsonl_file.stem}-{len(docs)}"

            docs.append({
                "id": doc_id,
                "text": text,
                "source_file": jsonl_file.name,
                "metadata": record
            })

if not docs:
    raise ValueError("没有读取到任何 JSONL 数据，请检查路径和文件内容。")

texts = [d["text"] for d in docs]

print(f"共加载 {len(texts)} 条知识记录")

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
).astype("float32")

# 建 FAISS 索引
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # 内积相似度，配合 normalized embeddings
index.add(embeddings)


共加载 1344 条知识记录


Batches: 100%|██████████| 42/42 [00:18<00:00,  2.30it/s]


In [ ]:
from pathlib import Path
import shutil
import json

# 1) 先把工作目录切到短英文路径（只做一次）
# 在 notebook 里可先执行：%cd C:\k

BASE = Path(".")                     # 当前目录 C:\k
OUTPUT_DIR = BASE / "build_index"    # 相对路径

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_index = OUTPUT_DIR / "knowledge_index.faiss"

# 2) Faiss 先写相对路径（解析后是 C:\k\faiss_safe\...）
faiss.write_index(index, str(final_index))

# # 3) 再复制到输出目录（也是相对路径）

with open(OUTPUT_DIR / "knowledge_meta.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

In [21]:
import gc
gc.collect()  # 手动触发垃圾回收，释放 GPU 显存

11114